# Ridge Baseline on GPU — Colab Master Notebook

End-to-end execution of the `data-ridge-baseline` pipeline with a vectorized closed-form alpha sweep on a Colab T4 GPU. Sweeping ~1000 L2 regularization values finishes in seconds because we eigendecompose `X.T @ X` once and reduce every additional alpha to an elementwise broadcast.

**How to run**
1. **Runtime → Change runtime type → T4 GPU** (or any GPU; T4 is free and more than enough).
2. **Runtime → Run all** (Ctrl/Cmd+F9).
3. Cell 1 installs a pinned dependency stack and forces a clean kernel restart so numpy and its C-extension consumers are aligned. You'll see "Your session crashed" — that's expected, not an error.
4. **Click Runtime → Run all once more.** Cell 1 detects the deps are already correct and skips straight to the pipeline. Wall time after restart is roughly 2–3 minutes (mostly the DANDI dataset download).
5. The last cell prints a single JSON blob you can copy-paste back.

**Pipeline**
1. Pinned install + auto-restart guard
2. Clone repo + import shared modules
3. Download the NLB MC_RTT dandiset
4. Preprocess into the canonical `processed_mc_rtt.npz` schema
5. Vectorized GPU ridge sweep across event budgets × ~1000 alphas
6. Save canonical JSON results + efficiency summary
7. Inline frontier figure
8. Copy-paste output summary

Project: **EE207 Neuromorphic BCI** · branch `data-ridge-baseline`

## 1. Pinned install + auto-restart

This cell does the heavy lifting. The first time it runs on a fresh runtime it installs the dependency stack and force-kills the kernel; the second time it detects the deps are already correct and continues without restarting.

In [ ]:
# Bulletproof dependency install for Colab. Pins a numpy 2.x-compatible BCI
# stack and then force-restarts the runtime exactly once so freshly installed
# C extensions load cleanly.
import os
import subprocess
import sys

REQUIRED = [
    "numpy==2.0.2",
    "scipy==1.13.1",
    "pandas==2.2.2",
    "scikit-learn==1.5.2",
    "matplotlib==3.9.2",
    "h5py==3.12.1",
    "hdmf==3.14.5",
    "pynwb==2.8.3",
    "pyyaml",
    "tqdm",
    "dandi==0.76.0",
    "torch",
]

def _all_imports_ok() -> bool:
    try:
        import numpy  # noqa: F401
        import scipy  # noqa: F401
        import pandas  # noqa: F401
        import pynwb  # noqa: F401
        import hdmf  # noqa: F401
        import dandi  # noqa: F401
        from dandi.download import download  # noqa: F401
        import nlb_tools.nwb_interface  # noqa: F401
        import torch  # noqa: F401
        return True
    except Exception:
        return False

if _all_imports_ok():
    print("dependencies already aligned — skipping install + restart.")
else:
    print("installing pinned BCI stack ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *REQUIRED]
    )
    print("installing nlb_tools without its stale pandas pin ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "nlb-tools==0.0.4"]
    )
    print("done. restarting runtime — re-run all cells when it reconnects.")
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)

## 2. Repo + imports

Clones `data-ridge-baseline` (a no-op if it already exists), changes into it, and pulls in the shared decoding modules.

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = "https://github.com/manrajmondair/neuromorphic-bci.git"
BRANCH   = "data-ridge-baseline"
REPO_DIR = Path("/content/neuromorphic-bci")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--prune"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

device = "cuda" if torch.cuda.is_available() else "cpu"
head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"python    : {sys.version.split()[0]}")
print(f"platform  : {platform.platform()}")
print(f"torch     : {torch.__version__}")
print(f"device    : {device}")
if device == "cuda":
    print(f"gpu       : {torch.cuda.get_device_name(0)}")
    print(f"cuda      : {torch.version.cuda}")
print(f"branch    : {BRANCH} @ {head}")

In [ ]:
import logging
from pathlib import Path

import numpy as np

from src.data.preprocess import preprocess_mc_rtt, save_processed, load_processed
from src.evaluation.efficiency_tracker import (
    compute_efficiency_summary, save_efficiency_json,
)
from src.evaluation.experiment_runner import save_json_results
from src.features.event_budget import restrict_to_event_budget

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)
print("imports OK")

## 3. Dataset

Downloads the NLB MC_RTT dandiset if it isn't already on disk, then runs the full preprocessing pipeline into the canonical `processed_mc_rtt.npz` schema. Re-runs are cheap because both steps short-circuit when their outputs exist.

In [ ]:
raw_dir        = Path("data/raw")
processed_path = Path("data/processed/processed_mc_rtt.npz")
raw_dir.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)

if not any(raw_dir.rglob("*.nwb")):
    print("downloading MC_RTT from DANDI ...")
    result = subprocess.run(
        [sys.executable, "scripts/download_mc_rtt.py", "--log-level", "INFO"],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("--- STDERR ---")
        print(result.stderr)
        raise RuntimeError(
            f"download_mc_rtt.py failed (exit {result.returncode}); see stderr above"
        )

if processed_path.is_file():
    print(f"loading cached {processed_path}")
    data = load_processed(processed_path)
else:
    print("running preprocessing pipeline ...")
    data = preprocess_mc_rtt(raw_dir=raw_dir, bin_size_ms=50)
    save_processed(data, processed_path)

num_bins, num_neurons = data["spike_counts"].shape
print(f"num_bins={num_bins}  num_neurons={num_neurons}")
print(
    f"train={data['train_idx'].size}  "
    f"val={data['val_idx'].size}  "
    f"test={data['test_idx'].size}"
)

## 4. Vectorized GPU ridge sweep

Closed-form ridge with broadcasting over alpha. For training matrix `X ∈ R^(n×N)` and targets `y ∈ R^(n×K)`:

$$
W_\alpha = (X^\top X + \alpha I)^{-1} X^\top y
$$

Eigendecompose `X^T X = V diag(λ) V^T` once. Then for every candidate `α` the inverse is just `V diag((λ + α)^{-1}) V^T`. With `z = V^T X^T y` and `u = X_eval V`, predictions for the entire alpha grid become

$$
\hat y_\alpha = u \cdot \frac{z}{\lambda + \alpha}.
$$

That's `O(N^3)` once plus `O(A · m · N)` for `A` alphas — milliseconds for `A ≈ 1000` on T4.

In [ ]:
def batch_ridge_predict(
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_eval:  torch.Tensor,
    alphas:  torch.Tensor,
) -> torch.Tensor:
    """Closed-form ridge predictions for many alphas in a single GPU pass.

    X_train : [n, N]      training features
    y_train : [n, K]      training targets
    X_eval  : [m, N]      features to predict on
    alphas  : [A]         L2 regularization grid

    Returns : [A, m, K]   prediction tensor across all alphas.
    """
    XtX = X_train.T @ X_train                          # [N, N]
    Xty = X_train.T @ y_train                          # [N, K]
    evals, evecs = torch.linalg.eigh(XtX)              # symmetric PSD
    z = evecs.T @ Xty                                  # [N, K]
    u = X_eval @ evecs                                 # [m, N]
    denom = evals.unsqueeze(0) + alphas.unsqueeze(1)    # [A, N]
    filtered = z.unsqueeze(0) / denom.unsqueeze(-1)     # [A, N, K]
    return torch.einsum("mn,ank->amk", u, filtered)     # [A, m, K]


def joint_r2_per_alpha(y_true: torch.Tensor, preds: torch.Tensor) -> torch.Tensor:
    """Joint R^2 across both velocity axes, vectorized over alphas."""
    mean_v = y_true.mean(dim=0, keepdim=True)
    ss_tot = ((y_true - mean_v) ** 2).sum()
    ss_res = ((preds - y_true.unsqueeze(0)) ** 2).sum(dim=(1, 2))  # [A]
    return 1.0 - ss_res / ss_tot

## 5. Sweep across event budgets × ~1000 alphas

Single deterministic seed — ridge with a fixed train/val/test split and the earliest-events budget filter is closed-form deterministic, so additional seeds reproduce identical results bit-for-bit.

In [ ]:
import time

EVENT_BUDGETS = (1.00, 0.50, 0.25, 0.10)
SEEDS         = (0,)
N_ALPHAS      = 1000
ALPHA_LO_HI   = (1e-4, 1e5)
DTYPE         = torch.float64   # eigendecomp is well-conditioned in float64

alphas    = torch.logspace(
    np.log10(ALPHA_LO_HI[0]), np.log10(ALPHA_LO_HI[1]),
    N_ALPHAS, device=device, dtype=DTYPE,
)
y         = torch.tensor(np.asarray(data["velocity"], dtype=np.float64), device=device, dtype=DTYPE)
train_idx = torch.tensor(data["train_idx"], device=device)
val_idx   = torch.tensor(data["val_idx"],   device=device)
test_idx  = torch.tensor(data["test_idx"],  device=device)
n_events_total = int(sum(t.size for t in data["event_times"]))

rows = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    for f in EVENT_BUDGETS:
        sub = restrict_to_event_budget(data, fraction=f)
        X = torch.tensor(sub["spike_counts"].astype(np.float64), device=device, dtype=DTYPE)

        t0 = time.time()
        preds_val = batch_ridge_predict(X[train_idx], y[train_idx], X[val_idx], alphas)
        r2_val    = joint_r2_per_alpha(y[val_idx], preds_val)
        best_a    = int(torch.argmax(r2_val).item())
        best_alpha = float(alphas[best_a].item())

        preds_test = batch_ridge_predict(
            X[train_idx], y[train_idx], X[test_idx], alphas[best_a:best_a+1],
        )[0]
        elapsed = time.time() - t0

        y_test = y[test_idx]
        ss_res_ax = ((preds_test - y_test) ** 2).sum(dim=0).cpu().numpy()
        ss_tot_ax = ((y_test - y_test.mean(0, keepdim=True)) ** 2).sum(dim=0).cpu().numpy()
        r2_ax     = 1.0 - ss_res_ax / np.where(ss_tot_ax == 0, 1.0, ss_tot_ax)
        r2_joint  = 1.0 - ss_res_ax.sum() / ss_tot_ax.sum()

        n_events_used = int(sum(t.size for t in sub["event_times"]))
        rows.append({
            "model":          "ridge",
            "event_budget":   float(f),
            "seed":           int(seed),
            "r2_vx":          float(r2_ax[0]),
            "r2_vy":          float(r2_ax[1]),
            "r2_joint":       float(r2_joint),
            "best_alpha":     best_alpha,
            "n_events_used":  n_events_used,
            "n_events_total": n_events_total,
            "notes":          f"gpu sweep over {N_ALPHAS} alphas in {elapsed:.2f}s",
        })
        print(
            f"seed={seed} f={f:.2f}  best α={best_alpha:.4g}  "
            f"r2_joint={r2_joint:+.4f}  vx={r2_ax[0]:+.4f}  vy={r2_ax[1]:+.4f}  "
            f"({elapsed:.2f}s for {N_ALPHAS} alphas)"
        )

## 6. Save canonical results + efficiency summary

Writes both files using the exact schema the local `scripts/generate_final_figures.py` already consumes. Once the SNN side lands, the same JSONs overlay automatically.

In [ ]:
results_dir = Path("results/ridge")
results_dir.mkdir(parents=True, exist_ok=True)

config = {
    "processed_path":   str(processed_path),
    "bin_size_ms":      int(data["bin_size_ms"]),
    "num_neurons":      int(data["num_neurons"]),
    "event_budgets":    list(EVENT_BUDGETS),
    "seeds":            list(SEEDS),
    "n_alphas":         int(N_ALPHAS),
    "alpha_lo_hi":      list(ALPHA_LO_HI),
    "device":           device,
    "dtype":            str(DTYPE),
    "notebook":         "02_ridge_baseline_colab.ipynb",
}
save_json_results(
    results_dir / "ridge_results.json",
    model="ridge",
    config=config,
    rows=rows,
)
save_efficiency_json(
    compute_efficiency_summary(data, fractions=EVENT_BUDGETS),
    results_dir / "computational_efficiency.json",
)
print("wrote:")
print(f"  {results_dir/'ridge_results.json'}")
print(f"  {results_dir/'computational_efficiency.json'}")

## 7. Quick frontier visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df  = pd.DataFrame(rows)
agg = df.groupby("event_budget")["r2_joint"].agg(["mean", "std"]).reset_index()
agg = agg.sort_values("event_budget", ascending=False)
agg["std"] = agg["std"].fillna(0.0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(
    agg["event_budget"], agg["mean"], yerr=agg["std"],
    marker="o", linewidth=2, capsize=4, label="Ridge (GPU sweep)",
)
ax.axhline(0.0, color="black", linewidth=0.5, alpha=0.4)
ax.set_xlim(1.05, -0.02)
ax.set_ylim(-0.05, 1.0)
ax.set_xlabel("Event budget f")
ax.set_ylabel("Velocity R² (joint)")
ax.set_title("Ridge baseline on Colab GPU — accuracy vs. event budget")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## 8. Copy-paste output summary

Run this last. The single JSON blob below contains everything I (Manraj) need to share back — pipeline metadata, every (event budget, seed) result row, and the efficiency table. Copy the whole block and paste it back into the chat.

In [ ]:
import json

import pandas as pd

summary = {
    "branch":         BRANCH,
    "commit":         head,
    "device":         device,
    "gpu":            torch.cuda.get_device_name(0) if device == "cuda" else None,
    "torch":          torch.__version__,
    "num_bins":       int(num_bins),
    "num_neurons":    int(num_neurons),
    "n_alphas":       int(N_ALPHAS),
    "alpha_lo_hi":    list(ALPHA_LO_HI),
    "event_budgets":  list(EVENT_BUDGETS),
    "seeds":          list(SEEDS),
    "ridge_rows":     rows,
    "efficiency":     compute_efficiency_summary(data, fractions=EVENT_BUDGETS),
}

print("===== COPY EVERYTHING BELOW THIS LINE =====")
print(json.dumps(summary, indent=2))
print("===== COPY EVERYTHING ABOVE THIS LINE =====")